# MNIST con Keras

Arquitectura común: **784 → 128 (ReLU) → 64 (ReLU) → 10 (softmax)**. Adam (`lr=0.001`), entropía cruzada, batch size 64, 10 épocas y semilla 42.

In [ ]:
from pathlib import Path
import json
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from tensorflow import keras
from tensorflow.keras import layers

ROOT = Path.cwd().resolve()
if ROOT.name == 'src':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from mnist_loader import load_mnist

SEED = 42
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except RuntimeError:
    pass

In [ ]:
X_train, y_train, X_test, y_test = load_mnist(ROOT / 'data')
assert X_train.shape == (60000, 784)
assert y_train.shape == (60000,)
assert X_test.shape == (10000, 784)
assert y_test.shape == (10000,)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for image, label, axis in zip(X_train[:10], y_train[:10], axes.flat):
    axis.imshow(image.reshape(28, 28), cmap='gray')
    axis.set_title(f'Etiqueta: {label}')
    axis.axis('off')
plt.tight_layout()
plt.show()

X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0
y_train = y_train.astype(np.int64)
y_test = y_test.astype(np.int64)

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
start = time.perf_counter()
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
)
training_time = time.perf_counter() - start
loss_curve = history.history['loss']

In [ ]:
test_loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
y_pred = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_test, y_pred)
print(f'Accuracy test: {accuracy:.4%}')
print(f'Tiempo de entrenamiento: {training_time:.2f} s')
assert accuracy > 0.90, 'Accuracy inesperadamente baja; revisar el preprocesamiento o entrenamiento.'

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(range(1, EPOCHS + 1), loss_curve, marker='o')
axes[0].set(title='Pérdida — Keras', xlabel='Época', ylabel='Cross-entropy')
axes[0].grid(alpha=0.3)
ConfusionMatrixDisplay(cm).plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Matriz de confusión — Keras')
plt.tight_layout()
plt.show()

results_dir = ROOT / 'results'
results_dir.mkdir(exist_ok=True)
result = {'framework': 'Keras', 'accuracy': float(accuracy), 'training_time_seconds': training_time, 'loss': loss_curve, 'confusion_matrix': cm.tolist()}
with (results_dir / 'keras.json').open('w', encoding='utf-8') as file:
    json.dump(result, file, indent=2)